[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

# Example 2.4 — Four-Bar Mechanism Synthesis via Genetic Algorithm + Nelder-Mead

Given a **target coupler-point trajectory** generated from a known mechanism, we recover the four link lengths $(r_3, r_4, r_p, \delta)$ that reproduce it exactly.

**Two-phase approach:**
1. **Genetic Algorithm (GA)** — global search to escape local minima.
2. **Nelder-Mead simplex** — derivative-free local refinement to drive error to zero.

**Reference:** Chapter 2 — *Four-Bar Mechanisms: Design with GA and Neural Networks*

## 1. Kinematics — Coupler-Point Trajectory

In [ ]:
from numpy import *

# ─── Fixed parameters ─────────────────────────────────────────────────
R1 = 120.0   # ground link length [mm]  (fixed reference)
R2 = 40.0    # crank length       [mm]  (fixed reference)
N  = 48      # number of crank positions

# ─── True mechanism parameters (synthesis target) ──────────────────────
R3_TRUE  = 100.0        # coupler length [mm]
R4_TRUE  =  80.0        # rocker length  [mm]
RP_TRUE  =  60.0        # coupler point distance [mm]
D_TRUE   = pi / 4.0     # coupler point angle [rad]

def coupler_curve(r3, r4, rp, delta, r1=R1, r2=R2, n=N):
    """
    Compute coupler-point trajectory using the vector-loop method.
    Returns (Px, Py) arrays of length n, or None if mechanism is not viable.
    """
    th2  = linspace(0, 2*pi, n, endpoint=False)
    Bx   = r2 * cos(th2)
    By   = r2 * sin(th2)
    BDx  = r1 - Bx
    BDy  = -By
    BD   = sqrt(BDx**2 + BDy**2)
    # Viability: triangle inequality for the four-bar loop
    if any(BD > r3 + r4 + 1e-9) or any(BD < abs(r3 - r4) - 1e-9):
        return None
    phi  = arctan2(BDy, BDx)
    cos_a = clip((r3**2 + BD**2 - r4**2) / (2.0 * r3 * BD), -1.0, 1.0)
    th3  = phi + arccos(cos_a)           # coupler angle
    Px   = Bx + rp * cos(th3 + delta)
    Py   = By + rp * sin(th3 + delta)
    return Px, Py

# ─── Generate target trajectory from the true mechanism ────────────────
TARGET = coupler_curve(R3_TRUE, R4_TRUE, RP_TRUE, D_TRUE)
Xt, Yt = TARGET
print(f'Target trajectory: {len(Xt)} points')
print(f'X range: [{Xt.min():.1f}, {Xt.max():.1f}] mm')
print(f'Y range: [{Yt.min():.1f}, {Yt.max():.1f}] mm')

# Verify true solution gives zero error
Px0, Py0 = coupler_curve(R3_TRUE, R4_TRUE, RP_TRUE, D_TRUE)
err0 = mean(sqrt((Px0 - Xt)**2 + (Py0 - Yt)**2))
print(f'True-solution error (must be 0): {err0:.2e} mm')


Target trajectory: 48 points
X range: [-23.2, 43.5] mm
Y range: [18.3, 94.8] mm
True-solution error (must be 0): 0.00e+00 mm


## 2. Fitness Function

In [ ]:
def fitness(params):
    """
    Mean point-to-point error between synthesized and target trajectories.
    params = [r3, r4, rp, delta]
    Returns error in mm (large penalty if mechanism is not viable).
    """
    r3, r4, rp, delta = params
    result = coupler_curve(r3, r4, rp, delta)
    if result is None:
        return 1e6
    Px, Py = result
    return mean(sqrt((Px - Xt)**2 + (Py - Yt)**2))

# Search space bounds for [r3, r4, rp, delta]
LB  = array([ 50.0,  30.0, 15.0, 0.0      ])
UB  = array([160.0, 140.0, 110.0, 2*pi    ])
RNG = UB - LB
print('Search bounds:')
print(f'  r3    : [{LB[0]:.0f}, {UB[0]:.0f}] mm')
print(f'  r4    : [{LB[1]:.0f}, {UB[1]:.0f}] mm')
print(f'  rp    : [{LB[2]:.0f}, {UB[2]:.0f}] mm')
print(f'  delta : [{LB[3]:.4f}, {UB[3]:.4f}] rad')


Search bounds:
  r3    : [50, 160] mm
  r4    : [30, 140] mm
  rp    : [15, 110] mm
  delta : [0.0000, 6.2832] rad


## 3. Phase 1 — Genetic Algorithm (Global Search)

The GA explores the full search space to avoid local minima. We run **5 independent seeds** (multi-start) and keep the best result.

**Operators:**
- Selection: tournament (size 4)
- Crossover: uniform blend
- Mutation: Gaussian, rate 12%, σ decreases from 10% → 3% of range after gen 300
- Elitism: top 6 individuals survive unchanged

In [ ]:
def run_ga(seed, n_pop=250, n_gen=600, early_stop=0.05):
    """
    Single GA run with the given random seed.
    Returns (best_individual, best_fitness).
    """
    random.seed(seed)
    pop = random.rand(n_pop, 4) * RNG + LB     # random initial population

    for gen in range(n_gen):
        fa  = array([fitness(ind) for ind in pop])
        idx = argsort(fa)
        if fa[idx[0]] < early_stop:
            break

        # Elitism: keep top 6
        new_pop = list(pop[idx[:6]])

        # Fill remaining slots
        while len(new_pop) < n_pop:
            # Tournament selection (4 candidates, pick best)
            t1 = random.choice(n_pop, 4, replace=False)
            t2 = random.choice(n_pop, 4, replace=False)
            p1 = pop[t1[argmin(fa[t1])]].copy()
            p2 = pop[t2[argmin(fa[t2])]].copy()

            # Uniform-blend crossover
            alpha = random.rand(4)
            child = alpha * p1 + (1 - alpha) * p2

            # Gaussian mutation
            sigma = 0.10 if gen < 300 else 0.03
            for j in range(4):
                if random.rand() < 0.12:
                    child[j] += random.randn() * sigma * RNG[j]
            new_pop.append(clip(child, LB, UB))

        pop = array(new_pop[:n_pop])

    fa   = array([fitness(ind) for ind in pop])
    best = pop[argmin(fa)]
    return best, fitness(best)

# ─── Multi-start GA (5 seeds) ───────────────────────────────────────────
import time
print('Running multi-start GA (5 seeds) ...')
t0 = time.time()
all_ga = []
for seed in [7, 13, 42, 99, 137]:
    best_ga, err_ga = run_ga(seed)
    all_ga.append((err_ga, best_ga, seed))
    print(f'  seed={seed:3d}: error={err_ga:.6f} mm  '
          f'r3={best_ga[0]:.2f}  r4={best_ga[1]:.2f}  '
          f'rp={best_ga[2]:.2f}  delta={best_ga[3]:.4f}')

all_ga.sort(key=lambda x: x[0])
ga_err, ga_best, ga_seed = all_ga[0]
print(f'\nBest GA result: seed={ga_seed},  error={ga_err:.6f} mm')
print(f'Elapsed: {time.time()-t0:.1f} s')


Running multi-start GA (5 seeds) ...
  seed=  7: error=0.067461 mm  r3=100.06  r4=81.47  rp=60.00  delta=0.7692
  seed= 13: error=0.044349 mm  r3=100.15  r4=79.84  rp=60.00  delta=0.7866
  seed= 42: error=0.078171 mm  r3=100.08  r4=81.70  rp=60.00  delta=0.7666
  seed= 99: error=0.064815 mm  r3=100.08  r4=81.40  rp=60.00  delta=0.7699
  seed=137: error=0.054360 mm  r3=100.05  r4=81.17  rp=60.00  delta=0.7724

Best GA result: seed=13,  error=0.044349 mm
Elapsed: 124.9 s


## 4. Phase 2 — Nelder-Mead Simplex Refinement

The Nelder-Mead simplex method is a **derivative-free** local optimizer.
It builds a simplex of $n+1$ points in parameter space and moves it toward the minimum through reflection, expansion, contraction, and shrinkage.

Key advantage over gradient descent: it uses a **geometric step size** (set to 1 mm / 1 rad) that is large enough to navigate the flat valley near the GA solution and reach the true minimum.

Convergence criterion: $E < 10^{-10}$ mm.

In [ ]:
def nelder_mead(f, x0, step=1.0, tol=1e-10, max_iter=30000):
    """
    Pure-NumPy Nelder-Mead simplex minimizer.

    Parameters
    ----------
    f        : objective function f(x) -> float
    x0       : initial point (array of length n)
    step     : initial simplex size (default 1.0)
    tol      : convergence tolerance on f(x)
    max_iter : maximum number of iterations

    Returns
    -------
    x_best : best parameter vector found
    f_best : best fitness value
    iters  : number of iterations performed
    """
    n = len(x0)
    # Build initial simplex: one vertex per parameter axis
    S = zeros((n + 1, n))
    S[0] = x0.copy()
    for i in range(n):
        v = x0.copy(); v[i] += step; S[i + 1] = v
    fv = array([f(s) for s in S])

    for it in range(max_iter):
        # Sort vertices by fitness
        o = argsort(fv); S = S[o]; fv = fv[o]
        if fv[0] < tol:
            break

        # Centroid of all but the worst vertex
        xb = mean(S[:-1], axis=0)

        # Reflection
        xr = xb + 1.0 * (xb - S[-1]); fr = f(xr)

        if fr < fv[0]:
            # Expansion
            xe = xb + 2.0 * (xr - xb); fe = f(xe)
            if fe < fr: S[-1] = xe; fv[-1] = fe
            else:       S[-1] = xr; fv[-1] = fr
        elif fr < fv[-2]:
            # Accept reflection
            S[-1] = xr; fv[-1] = fr
        else:
            # Contraction
            if fr < fv[-1]:
                xc = xb + 0.5 * (xr - xb); fc = f(xc)
                if fc <= fr: S[-1] = xc; fv[-1] = fc
                else:
                    S[1:] = S[0] + 0.5 * (S[1:] - S[0])
                    fv[1:] = array([f(s) for s in S[1:]])
            else:
                xc = xb + 0.5 * (S[-1] - xb); fc = f(xc)
                if fc < fv[-1]: S[-1] = xc; fv[-1] = fc
                else:
                    S[1:] = S[0] + 0.5 * (S[1:] - S[0])
                    fv[1:] = array([f(s) for s in S[1:]])

    return S[0], fv[0], it + 1

# ─── Apply Nelder-Mead to GA best solution ──────────────────────────────
print('Applying Nelder-Mead refinement ...')
t1 = time.time()
nm_best, nm_err, nm_iters = nelder_mead(fitness, ga_best.copy(),
                                         step=1.0, tol=1e-10)
print(f'NM result  : {nm_err:.2e} mm  ({nm_iters} iterations, {time.time()-t1:.2f} s)')
print(f'Parameters : r3={nm_best[0]:.6f}  r4={nm_best[1]:.6f}  '
      f'rp={nm_best[2]:.6f}  delta={nm_best[3]:.6f}')
print(f'True values: r3={R3_TRUE:.6f}  r4={R4_TRUE:.6f}  '
      f'rp={RP_TRUE:.6f}  delta={D_TRUE:.6f}')


Applying Nelder-Mead refinement ...
NM result  : 8.20e-11 mm  (191 iterations, 0.07 s)
Parameters : r3=100.000000  r4=80.000000  rp=60.000000  delta=0.785398
True values: r3=100.000000  r4=80.000000  rp=60.000000  delta=0.785398


## 5. Results and Trajectory Comparison

In [ ]:
# ─── Print final summary ────────────────────────────────────────────────
print('=' * 58)
print('          SYNTHESIS RESULTS SUMMARY')
print('=' * 58)
print('  True solution error    : %.2e mm' % err0)
print('  After GA only          : %.6f mm' % ga_err)
print('  After GA + Nelder-Mead : %.2e mm' % nm_err)
print()
hdr = '  %-12s  %10s  %10s  %10s' % ('Parameter', 'True', 'Found', 'Error')
print(hdr)
print('  ' + '-' * 48)
names  = ['r3 [mm]', 'r4 [mm]', 'rp [mm]', 'delta [rad]']
trues  = [R3_TRUE, R4_TRUE, RP_TRUE, D_TRUE]
for nm, tv, fv in zip(names, trues, nm_best):
    row = '  %-12s  %10.4f  %10.4f  %10.2e' % (nm, tv, fv, abs(tv - fv))
    print(row)
print('=' * 58)

# Grashof condition check for found mechanism
links = [R1, R2, nm_best[0], nm_best[1]]
s_l, l_l = min(links), max(links)
p_l = sorted(links)[1]; q_l = sorted(links)[2]
grashof = 'YES (crank-rocker)' if s_l + l_l <= p_l + q_l else 'NO'
print('  Grashof satisfied: ' + grashof)


          SYNTHESIS RESULTS SUMMARY
  True solution error    : 0.00e+00 mm
  After GA only          : 0.044349 mm
  After GA + Nelder-Mead : 8.20e-11 mm

  Parameter           True       Found       Error
  ------------------------------------------------
  r3 [mm]         100.0000    100.0000    2.19e-10
  r4 [mm]          80.0000     80.0000    1.12e-09
  rp [mm]          60.0000     60.0000    1.03e-11
  delta [rad]       0.7854      0.7854    1.31e-11
  Grashof satisfied: YES (crank-rocker)


In [ ]:
# ─── Plot: target vs. synthesized trajectory ───────────────────────────
try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    Ps, Qs = coupler_curve(*nm_best)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Left: trajectory overlay
    ax = axes[0]
    ax.plot(Xt, Yt, 'b-o', ms=4, lw=2, label='Target (true mechanism)')
    ax.plot(Ps, Qs, 'r--s', ms=4, lw=1.5, label='Synthesized (GA+NM)')
    ax.set_xlabel('X [mm]', fontsize=12)
    ax.set_ylabel('Y [mm]', fontsize=12)
    ax.set_title('Coupler-Point Trajectory', fontsize=13)
    ax.legend(fontsize=10)
    ax.set_aspect('equal'); ax.grid(True, alpha=0.4)

    # Right: point-by-point error
    ax2 = axes[1]
    err_pt = sqrt((Ps - Xt)**2 + (Qs - Yt)**2)
    ax2.plot(range(N), err_pt * 1e3, 'g-o', ms=4, lw=1.5)
    ax2.set_xlabel('Crank position index', fontsize=12)
    ax2.set_ylabel('Point error [μm]', fontsize=12)
    ax2.set_title(f'Point-by-point Error  (mean={nm_err*1e3:.3f} μm)', fontsize=13)
    ax2.grid(True, alpha=0.4)

    fig.suptitle('Example 2.4 — Four-Bar Synthesis: GA + Nelder-Mead',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('ex2_4_result.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Figure saved: ex2_4_result.png')
except ImportError:
    print('matplotlib not available — skipping plot.')


Figure saved: ex2_4_result.png
